# Week 8 Evaluation Expansion

## Goal

This notebook evaluates the Week 7 XGBoost predictions in more detail.

It will:

1. load the Week 7 test predictions
2. calculate R², MAPE, and MdAPE
3. divide homes into low, middle, and high price groups
4. compare the model across the three price groups
5. save `metrics_summary.csv` in `results/06/`
6. print simple findings about which price groups perform better


## 1. Import the packages

Only basic packages are needed for this evaluation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import r2_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("Packages are ready.")

Packages are ready.


## 2. Set the file paths

The Week 7 prediction file is expected at:

`results/05/xgboost_test_predictions.csv`

The Week 8 output file will be saved in:

`results/06/metrics_summary.csv`


In [2]:
working_dir = Path.cwd()

project_root = (
    working_dir.parent
    if working_dir.name.lower() == "notebooks"
    else working_dir
)

predictions_path = (
    project_root
    / "results"
    / "05"
    / "xgboost_test_predictions.csv"
)

output_dir = project_root / "results" / "06"
output_dir.mkdir(parents=True, exist_ok=True)

metrics_output_path = output_dir / "metrics_summary.csv"

print("Input file:", predictions_path)
print("Output folder:", output_dir)

Input file: C:\Users\18327\IDX Exchange\results\05\xgboost_test_predictions.csv
Output folder: C:\Users\18327\IDX Exchange\results\06


## 3. Load the Week 7 predictions

The Week 7 notebook already saved the actual price and predicted price for every test row. We can use that file directly, so the model does not need to be trained again.

In [3]:
if not predictions_path.exists():
    raise FileNotFoundError(
        f"Cannot find {predictions_path}. "
        "Please make sure the Week 7 results are inside results/05/."
    )

predictions = pd.read_csv(predictions_path)

required_columns = ["Actual_Price", "Predicted_Price"]
missing_columns = [
    column
    for column in required_columns
    if column not in predictions.columns
]

if missing_columns:
    raise ValueError(
        "The prediction file is missing these columns: "
        f"{missing_columns}"
    )

predictions = predictions[
    predictions["Actual_Price"].notna()
    & predictions["Predicted_Price"].notna()
    & (predictions["Actual_Price"] != 0)
].copy()

print("Number of test rows:", len(predictions))
display(predictions.head())

Number of test rows: 4652


,Original_Row_Index,Actual_Price,Predicted_Price,Error,Absolute_Error,Absolute_Percentage_Error
0,16519,"710,000.0000","656,224.1000","-53,775.8750","53,775.8750",7.5741
1,6366,"2,735.0000","25,009.0040","22,274.0039","22,274.0039",814.4060
2,2602,"1,755,000.0000","1,649,520.4000","-105,479.6250","105,479.6250",6.0102
3,17636,"497,950.0000","489,212.6000","-8,737.4062","8,737.4062",1.7547
4,13084,"1,200,000.0000","1,136,084.2000","-63,915.7500","63,915.7500",5.3263


## 4. Define MAPE and MdAPE

MAPE is the mean absolute percentage error. It shows the average percentage error.

MdAPE is the median absolute percentage error. It shows the typical percentage error and is less affected by a few very large errors.

For both measures, a smaller value is better.

In [4]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    absolute_percentage_error = (
        np.abs(y_true - y_pred)
        / np.abs(y_true)
    )

    return 100 * np.mean(absolute_percentage_error)


def mdape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    absolute_percentage_error = (
        np.abs(y_true - y_pred)
        / np.abs(y_true)
    )

    return 100 * np.median(absolute_percentage_error)


print("Metric functions are ready.")

Metric functions are ready.


## 5. Calculate the overall evaluation

This section calculates R², MAPE, and MdAPE for the full Week 7 test set.

In [5]:
actual = predictions["Actual_Price"]
predicted = predictions["Predicted_Price"]

overall_r2 = r2_score(actual, predicted)
overall_mape = mape(actual, predicted)
overall_mdape = mdape(actual, predicted)

overall_metrics = pd.DataFrame(
    {
        "Metric": ["R2", "MAPE_Percent", "MdAPE_Percent"],
        "Value": [overall_r2, overall_mape, overall_mdape],
    }
)

display(overall_metrics)

print(f"Overall R²: {overall_r2:.4f}")
print(f"Overall MAPE: {overall_mape:.2f}%")
print(f"Overall MdAPE: {overall_mdape:.2f}%")

,Metric,Value
0,R2,-41.4827
1,MAPE_Percent,"2,042.2206"
2,MdAPE_Percent,9.8087


Overall R²: -41.4827
Overall MAPE: 2042.22%
Overall MdAPE: 9.81%


## 6. Create three price groups

To compare different price levels, the test homes are divided into three groups based on actual sale price.

The three groups contain about the same number of rows.

1. Low price
2. Middle price
3. High price


In [6]:
predictions["Price_Band"] = pd.qcut(
    predictions["Actual_Price"],
    q=3,
    labels=["Low price", "Middle price", "High price"],
)

price_band_ranges = (
    predictions
    .groupby("Price_Band", observed=True)["Actual_Price"]
    .agg(["count", "min", "max"])
    .reset_index()
)

price_band_ranges.columns = [
    "Price_Band",
    "Number_of_Rows",
    "Minimum_Actual_Price",
    "Maximum_Actual_Price",
]

display(price_band_ranges)

,Price_Band,Number_of_Rows,Minimum_Actual_Price,Maximum_Actual_Price
0,Low price,1551,1.5300,"389,000.0000"
1,Middle price,1550,"390,000.0000","927,000.0000"
2,High price,1551,"930,000.0000","28,000,000.0000"


## 7. Calculate metrics for each price group

The same R², MAPE, and MdAPE measures are calculated for each group.

In [7]:
summary_rows = [
    {
        "Group": "Overall",
        "Price_Range": "All test homes",
        "Number_of_Rows": len(predictions),
        "R2": overall_r2,
        "MAPE_Percent": overall_mape,
        "MdAPE_Percent": overall_mdape,
    }
]

for price_band, group_data in predictions.groupby(
    "Price_Band",
    observed=True,
):
    group_actual = group_data["Actual_Price"]
    group_predicted = group_data["Predicted_Price"]

    minimum_price = group_actual.min()
    maximum_price = group_actual.max()

    summary_rows.append(
        {
            "Group": str(price_band),
            "Price_Range": (
                f"${minimum_price:,.0f} to "
                f"${maximum_price:,.0f}"
            ),
            "Number_of_Rows": len(group_data),
            "R2": r2_score(
                group_actual,
                group_predicted,
            ),
            "MAPE_Percent": mape(
                group_actual,
                group_predicted,
            ),
            "MdAPE_Percent": mdape(
                group_actual,
                group_predicted,
            ),
        }
    )

metrics_summary = pd.DataFrame(summary_rows)

display(metrics_summary)

,Group,Price_Range,Number_of_Rows,R2,MAPE_Percent,MdAPE_Percent
0,Overall,All test homes,4652,-41.4827,"2,042.2206",9.8087
1,Low price,"$2 to $389,000",1551,-124.3804,"6,053.3458",498.7216
2,Middle price,"$390,000 to $927,000",1550,-270.2249,29.1239,7.9552
3,High price,"$930,000 to $28,000,000",1551,-68.4264,42.8942,4.4522


## 8. Save the Week 8 metrics summary

This is the required Week 8 CSV file.

In [8]:
metrics_summary.to_csv(
    metrics_output_path,
    index=False,
)

print("Saved:", metrics_output_path)

Saved: C:\Users\18327\IDX Exchange\results\06\metrics_summary.csv


## 9. Summarize the main findings

The code below finds the best and worst price groups using MAPE and MdAPE.

A lower MAPE or MdAPE means better percentage accuracy.

In [9]:
price_band_metrics = metrics_summary[
    metrics_summary["Group"] != "Overall"
].copy()

best_mape_row = price_band_metrics.loc[
    price_band_metrics["MAPE_Percent"].idxmin()
]

best_mdape_row = price_band_metrics.loc[
    price_band_metrics["MdAPE_Percent"].idxmin()
]

worst_mdape_row = price_band_metrics.loc[
    price_band_metrics["MdAPE_Percent"].idxmax()
]

print("Week 8 findings")
print()
print(
    f"Overall MAPE is {overall_mape:.2f}%."
)
print(
    f"Overall MdAPE is {overall_mdape:.2f}%."
)
print()
print(
    "Best average percentage accuracy by MAPE:",
    best_mape_row["Group"],
    f"with {best_mape_row['MAPE_Percent']:.2f}%."
)
print(
    "Best typical percentage accuracy by MdAPE:",
    best_mdape_row["Group"],
    f"with {best_mdape_row['MdAPE_Percent']:.2f}%."
)
print(
    "Worst typical percentage accuracy by MdAPE:",
    worst_mdape_row["Group"],
    f"with {worst_mdape_row['MdAPE_Percent']:.2f}%."
)

if overall_mape > 5 * overall_mdape:
    print()
    print(
        "MAPE is much larger than MdAPE. "
        "This means a small number of very large percentage errors "
        "are strongly increasing the average error."
    )

if predictions["Actual_Price"].min() < 10000:
    print()
    print(
        "The data includes some very small actual prices below $10,000. "
        "These rows can create extremely large percentage errors, "
        "especially in the low price group."
    )

Week 8 findings

Overall MAPE is 2042.22%.
Overall MdAPE is 9.81%.

Best average percentage accuracy by MAPE: Middle price with 29.12%.
Best typical percentage accuracy by MdAPE: High price with 4.45%.
Worst typical percentage accuracy by MdAPE: Low price with 498.72%.

MAPE is much larger than MdAPE. This means a small number of very large percentage errors are strongly increasing the average error.

The data includes some very small actual prices below $10,000. These rows can create extremely large percentage errors, especially in the low price group.


## 10. Results

The test set contains 4,652 properties.

The overall R² is -41.4827. This negative value means that the predictions have large errors for some properties, so the model does not perform well when R² is used for the full test set.

The overall MAPE is 2042.22%, while the overall MdAPE is only 9.81%. The very large difference between these two measures shows that a small number of properties have extremely large percentage errors. These extreme errors strongly increase MAPE, while MdAPE is less affected by them.

The price group results show clear differences.

For the low price group, MAPE is 6053.35% and MdAPE is 498.72%. This is the weakest price group. The data includes some very small actual prices, including prices below $10,000. When the actual price is very small, even a moderate prediction error can create a very large percentage error.

For the middle price group, MAPE is 29.12% and MdAPE is 7.96%. This group has the lowest MAPE, so it has the best average percentage accuracy among the three price groups.

For the high price group, MAPE is 42.89% and MdAPE is 4.45%. This group has the lowest MdAPE, so it has the best typical percentage accuracy.

Overall, the model performs much better for middle and high priced properties than for low priced properties.

## 11. Conclusion

The Week 8 evaluation shows that model performance changes across different property price levels.

The overall MAPE is very large because a small number of properties have extreme percentage errors. The overall MdAPE of 9.81% gives a more stable view of the typical prediction error.

The low price group performs the worst because some actual prices are very small. These small values make percentage based error measures very sensitive.

The middle price group performs best based on MAPE, with a MAPE of 29.12%. The high price group performs best based on MdAPE, with a MdAPE of 4.45%.

Therefore, the model is more reliable for middle and high priced properties than for low priced properties. For this dataset, MdAPE is especially useful because it is less affected by extreme observations.

The required file `metrics_summary.csv` is saved in `results/06/`.

In [10]:
print("Week 8 is complete.")
print("Required output:", metrics_output_path)

Week 8 is complete.
Required output: C:\Users\18327\IDX Exchange\results\06\metrics_summary.csv
